# Pulse-Level Compilation and Simulation of ECD and SNAP Circuits

Gate-level optimization (`gate_optimization`) returns an *ideal* circuit: a list of
$\beta_i$ and qubit rotations, or a list of SNAP phases and displacements. This notebook
takes that output and does the second half of the two-step optimization of
Eickbusch et al., *Nat. Phys.* **18**, 1464 (2022): compile each gate into a physical
control pulse, build the time-dependent Hamiltonian, propagate it, and read out the
Wigner function along the way.

The pipeline is

$$
\texttt{GateOptResult} \;\xrightarrow{\;\texttt{pulses}\;}\;
\bigl(\varepsilon(t),\,\Omega(t)\bigr) \;\xrightarrow{\;\texttt{pulse\_simulation}\;}\;
\lvert\tilde\psi(t)\rangle \;\xrightarrow{\;\texttt{utils}\;}\; W(x,p;t) .
$$

Three things are worth knowing before running anything.

**The simulation lives in a displaced frame.** ECD control drives the oscillator to
$|\alpha|^2 \sim 900$ photons, so a lab-frame simulation would need `n_fock` of order
1500. Instead the classical response $\alpha(t)$ is solved exactly and factored out,
$\lvert\psi\rangle = D(\alpha(t))\lvert\tilde\psi(t)\rangle$, leaving a residual state
that stays within a few photons of the origin. Lab-frame Wigner functions are then a
*coordinate shift* of the co-moving ones.

**The frame does not close.** $\alpha(T)\neq 0$, because the compiled pulse amplitudes
null the mean of the two *conditional* trajectories while $\alpha(t)$ follows the
ground branch alone. Fidelities must be evaluated after applying $D(\alpha(T))$;
skipping it costs a few parts in $10^4$.

**Compilation is slow and non-differentiable.** Each distinct $|\beta|$ costs a
root-find over a few hundred ODE solves, of order 10-25 s. Results are cached by
$|\beta|$ within a sequence, since the pulse for any $\beta$ follows from the one for
$|\beta|$ by a global phase rotation.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np

import jax
import jax.numpy as jnp

from IPython.display import HTML

from gkp_optimal_control import pulses, pulse_simulation
from gkp_optimal_control.animation import animate_wigner
from gkp_optimal_control.gate_optimization import (
    GateBounds,
    OptimizerConfig,
    optimize_gate_sequence,
)
from gkp_optimal_control.plotting import plot_wigner, set_plot_style
from gkp_optimal_control.states import gkp_states
from gkp_optimal_control.utils import to_ket

set_plot_style()

## System parameters

`SystemParams` defaults to Table S1 of the supplement. Every field is overridable, and
`.replace()` returns a modified copy. Units are microseconds and rad/us throughout, so a
rate quoted in Hz carries a factor $2\pi$.

The dispersive shift sets the natural gate timescale: a conditional displacement built
from the dispersive interaction alone takes $\pi/\chi$, while ECD control reaches the
same $\beta$ in a few hundred nanoseconds by first displacing the oscillator to a large
$\alpha_0$, where the *rate* of conditional phase-space separation is $\chi\alpha_0$
rather than $\chi$.

In [ ]:
params, cfg = pulses.load_config("./../devices/device_snap.toml")

print(f"chi / 2pi       = {params.chi / (2 * np.pi) * 1e3:8.2f} kHz")
print(f"chi' / 2pi      = {params.chi_prime / (2 * np.pi) * 1e6:8.2f} Hz")
print(f"K_c / 2pi       = {params.kerr / (2 * np.pi) * 1e6:8.2f} Hz")
print(f"T1 cavity       = {1 / params.kappa:8.1f} us")
print(f"2 pi / chi      = {2 * np.pi / params.chi:8.2f} us   <- dispersive gate timescale")
print(f"displacement    = {params.t_disp * 1e3:8.1f} ns   (sigma = {params.sigma_disp * 1e3:.0f} ns)")
print(f"qubit rotation  = {params.t_qubit * 1e3:8.1f} ns   (sigma = {params.sigma_qubit * 1e3:.0f} ns)")
print(f"eps_max / 2pi   = {params.eps_max / (2 * np.pi):8.1f} MHz")
print(f"Omega_max / 2pi = {params.omega_max / (2 * np.pi):8.1f} MHz")
print(f"sample period   = {params.dt * 1e3:8.1f} ns")

## Anatomy of one ECD gate

The pulse follows Fig. S3a: four Gaussian cavity displacements with a qubit $\pi$ pulse
at the midpoint, separated by wait times $t_w$ during which the two qubit-conditioned
coherent states rotate at $\pm\chi/2$ and so separate in phase space.

The four amplitudes are not equal. Their ratios are found by Nelder-Mead on the cost
function (S27), evaluated along the semiclassical trajectories (S5), which requires
that (i) the *mean* of the two conditional trajectories returns to the origin at $T/2$
and at $T$, and (ii) the intermediate radius equals $\alpha_0$ during both wait windows.
Second-order dispersive shift, Kerr and photon loss all enter through those trajectories,
so the compiled ratios are asymmetric.

`optimize_ecd_pulse` then reduces $t_w$ until $|\beta|$ matches the request. Because
$t_w$ is an integer number of DAC samples and one sample is worth roughly a percent of
$|\beta|$, the last step trims the radius instead: gates run at $\alpha_0' \le \alpha_0$,
exactly as in the experiment.

In [ ]:
beta_demo = 2.0 + 1.0j
alpha0 = 30.0

t0 = time.time()
pulse = pulses.optimize_ecd_pulse(beta_demo, alpha0, params, verbose=True)
print(f"\ncompiled in {time.time() - t0:.1f} s")
print(pulse.summary())
print(f"amplitude ratios [eps0, r2, r3, r4] = {pulse.ratios}")
print(f"gate duration    = {pulse.eps.size * params.dt * 1e3:.0f} ns")
print(f"speedup vs pi/chi = {np.pi / params.chi / (pulse.eps.size * params.dt):.0f}x")

In [ ]:
t = (np.arange(pulse.eps.size) + 0.5) * params.dt * 1e3  # ns
alpha_frame = pulses.frame_trajectory(pulse.eps, params, delta=0.5 * params.chi)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

ax = axes[0, 0]
ax.plot(t, np.real(pulse.eps) / (2 * np.pi), label=r"Re $\varepsilon$")
ax.plot(t, np.imag(pulse.eps) / (2 * np.pi), label=r"Im $\varepsilon$")
ax.set_xlabel("t (ns)")
ax.set_ylabel(r"$\varepsilon / 2\pi$ (MHz)")
ax.set_title("cavity drive")
ax.legend()

ax = axes[0, 1]
ax.plot(t, np.real(pulse.omega) / (2 * np.pi), label=r"Re $\Omega$")
ax.plot(t, np.imag(pulse.omega) / (2 * np.pi), label=r"Im $\Omega$")
ax.set_xlabel("t (ns)")
ax.set_ylabel(r"$\Omega / 2\pi$ (MHz)")
ax.set_title(r"transmon drive ($\pi$ pulse)")
ax.legend()

ax = axes[1, 0]
ax.plot(t, np.abs(pulse.alpha_g[:-1]), label=r"$|\alpha_g|$")
ax.plot(t, np.abs(pulse.alpha_e[:-1]), "--", label=r"$|\alpha_e|$")
ax.plot(t, np.abs(alpha_frame[:-1]), ":", label=r"$|\alpha|$ (frame)")
ax.axhline(pulse.alpha0, color="k", lw=0.8, ls="-.", label=r"$\alpha_0'$")
ax.set_xlabel("t (ns)")
ax.set_ylabel("phase-space radius")
ax.set_title("conditional trajectories")
ax.legend()

ax = axes[1, 1]
ax.plot(np.real(pulse.alpha_g), np.imag(pulse.alpha_g), label=r"$\alpha_g$")
ax.plot(np.real(pulse.alpha_e), np.imag(pulse.alpha_e), "--", label=r"$\alpha_e$")
ax.plot([0], [0], "k+", ms=10)
ax.set_aspect("equal")
ax.set_xlabel(r"Re $\alpha$")
ax.set_ylabel(r"Im $\alpha$")
ax.set_title(r"phase space (separation at $T$ is $\beta$)")
ax.legend()

fig.tight_layout()
plt.show()

print(f"beta realized = {pulse.beta:+.6f}   target {beta_demo:+.6f}")
print(f"residual net displacement |lambda| = {abs(pulse.lam):.2e}")
print(f"frame endpoint |alpha(T)|          = {abs(alpha_frame[-1]):.4f}  <- frame does not close")

## Gate time versus $|\beta|$

Reproduces the trade-off of Fig. 2c. For small $|\beta|$ the gate is *drive-constrained*:
$t_w$ has already collapsed to zero and the pulse cannot be shortened below the four
displacements plus the $\pi$ pulse, so the compiler lowers $\alpha_0'$ instead. For large
$|\beta|$ the wait time dominates and the duration grows roughly as
$\arcsin(|\beta| / 2\alpha_0) \, / \chi$.

Each distinct $|\beta|$ is a fresh root-find, so this cell takes a couple of minutes.

In [ ]:
beta_mags = np.array([0.25, 0.5, 0.8, 1.2, 1.8, 2.5, 3.2])
make_pulse = pulses.ecd_pulse_cache(params, alpha0=alpha0)

durations, radii, theta_primes = [], [], []
t0 = time.time()
for mag in beta_mags:
    p_i = make_pulse(mag + 0j)
    durations.append(p_i.eps.size * params.dt * 1e3)
    radii.append(p_i.alpha0)
    theta_primes.append(p_i.theta_prime)
    print(f"|beta| = {mag:4.2f}: T = {durations[-1]:5.0f} ns, "
          f"alpha0' = {radii[-1]:5.2f}, t_w = {p_i.t_wait * 1e3:5.1f} ns, "
          f"theta' = {p_i.theta_prime:+.4f}")
print(f"\ntotal {time.time() - t0:.0f} s")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].plot(beta_mags, durations, "o-")
axes[0].set_xlabel(r"$|\beta|$")
axes[0].set_ylabel("gate duration (ns)")
axes[0].set_title("ECD gate time")
axes[1].plot(beta_mags, radii, "o-")
axes[1].axhline(alpha0, color="k", lw=0.8, ls="--")
axes[1].set_xlabel(r"$|\beta|$")
axes[1].set_ylabel(r"$\alpha_0'$ used")
axes[1].set_title("drive-constrained at small " + r"$|\beta|$")
axes[2].plot(beta_mags, np.abs(theta_primes), "o-")
axes[2].set_xlabel(r"$|\beta|$")
axes[2].set_ylabel(r"$|\theta'|$ (rad)")
axes[2].set_title(r"geometric qubit phase $\approx 0.013\,|\beta|^2$")
fig.tight_layout()
plt.show()

## A target state and its gate-level circuit

Square-lattice GKP $\lvert 0_L\rangle$ with a finite-energy envelope. The circuit is
optimized at the gate level first; only then is it compiled. Set `RUN_OPTIMIZATION` to
`False` to reuse a stored parameter set and skip straight to compilation.

In [ ]:
from gkp_optimal_control.gate_results import gate_result_path, load_gate_result, save_gate_result
RUN_OPTIMIZATION = True

n_fock_gate = 60
delta_env = 0.30

gkp_0, gkp_1 = gkp_states(
    n_fock=n_fock_gate,
    alpha=np.sqrt(np.pi / 2),
    beta=1j * np.sqrt(np.pi / 2),
    delta=delta_env,
    cutoff=6,
)
psi_target = np.asarray(to_ket(gkp_0)).reshape(-1)
psi_vac = np.zeros(n_fock_gate, dtype=complex)
psi_vac[0] = 1.0

n_ecd = 16

_path = gate_result_path('data/gate_results/ecd_snap_circuit_pulse_compilation', 'ecd', n_ecd)
if RUN_OPTIMIZATION:
    result = optimize_gate_sequence(
        "ecd",
        n_gates=n_ecd,
        psi_init=psi_vac,
        psi_targ=psi_target,
        n_fock=n_fock_gate,
        loss_type="log_infidelity",
        bounds=GateBounds(max_disp=3.5, n_leak=5, leakage_weight=1.0),
        optimizer=OptimizerConfig(n_seeds=8, n_adam_iters=1500, seed=0),
        verbose=True,
    )
    save_gate_result(result, _path, initial_state=psi_vac, overwrite=True, source_notebook='ecd_snap_circuit_pulse_compilation')
    print()
    print(result.summary())
    betas = np.asarray(result.params["betas"])
    thetas = np.asarray(result.params["thetas"])
    phis = np.asarray(result.params["phis"])
elif _path.exists():
    result = load_gate_result(_path)
    print("loaded cached result from " + str(_path))
else:
    result = None
    betas = np.array([1.05 + 0.00j, -0.62 + 0.71j, 1.48 - 0.33j, 0.54 + 1.02j, -1.19 - 0.28j])
    thetas = np.array([1.571, 2.094, 1.047, 2.618, 0.785, 1.571])
    phis = np.array([0.000, 1.257, -0.628, 2.199, -1.885, 0.314])

print(f"\n|beta| = {np.abs(betas).round(3)}")

## Compiling the circuit

Time order is $R_1,\, \mathrm{ECD}_1,\, R_2,\, \dots,\, \mathrm{ECD}_N,\, R_{N+1}$,
matching `gate_optimization._build_ecd_sequence`.

Each ECD imparts a small extra qubit phase $\theta'$ (Eq. S21), so the realized gate is
$\mathrm{ECD}(\beta)\,Z(\theta')$. That phase is removed by *virtual Z*: the phase of each
rotation pulse is shifted by a running frame phase, at no cost in pulse time. The frame
does not simply accumulate. Because $Z(a)\sigma_x = \sigma_x Z(-a)$ and every ECD contains
a $\pi$ pulse, commuting the leftover $Z$ past a gate flips its sign, so the update is

$$
\varphi_i \to \varphi_i - F, \qquad F \to -F + \theta'_i .
$$

With this rule the compiled sequence reproduces the ideal circuit's
$\lvert g\rangle$-projected cavity state exactly for arbitrary $\theta'$; the naive
accumulating rule does not (see the frame-rule check).

In [ ]:
t0 = time.time()
seq = pulses.compile_ecd_sequence(
    betas, thetas, phis, alpha0=alpha0, params=params, verbose=True
)
print(f"\ncompiled in {time.time() - t0:.0f} s\n")
print(seq.summary())

limits = seq.check_drive_limits(warn=False)
print(f"\ndrive headroom: |eps| {limits['eps_peak'] / limits['eps_limit']:.2%} of limit, "
      f"|Omega| {limits['omega_peak'] / limits['omega_limit']:.2%} of limit")
print(f"theta' per gate : {np.round(seq.meta['theta_primes'], 4)}")
print(f"alpha0' per gate: {np.round(seq.meta['alpha0_used'], 2)}")
print(f"|beta| error    : {np.abs(np.abs(seq.meta['betas_realized']) - np.abs(betas)).max():.2e}")

In [ ]:
t_ns = seq.t * 1e3
alpha_seq = seq.frame_trajectory()

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

colors = {"rotation": "tab:orange", "ecd": "tab:blue", "displacement": "tab:green",
          "snap": "tab:purple", "idle": "0.9"}
for s in seq.segments:
    for ax in axes:
        ax.axvspan(s.start * params.dt * 1e3, s.stop * params.dt * 1e3,
                   color=colors.get(s.kind, "0.9"), alpha=0.12, lw=0)

axes[0].plot(t_ns, np.real(seq.eps) / (2 * np.pi), lw=0.8, label=r"Re $\varepsilon$")
axes[0].plot(t_ns, np.imag(seq.eps) / (2 * np.pi), lw=0.8, label=r"Im $\varepsilon$")
axes[0].set_ylabel(r"$\varepsilon/2\pi$ (MHz)")
axes[0].legend(loc="upper right")

axes[1].plot(t_ns, np.real(seq.omega) / (2 * np.pi), lw=0.8, label=r"Re $\Omega$")
axes[1].plot(t_ns, np.imag(seq.omega) / (2 * np.pi), lw=0.8, label=r"Im $\Omega$")
axes[1].set_ylabel(r"$\Omega/2\pi$ (MHz)")
axes[1].legend(loc="upper right")

axes[2].plot(t_ns, np.abs(alpha_seq[:-1]), lw=0.9)
axes[2].set_ylabel(r"$|\alpha(t)|$")
axes[2].set_xlabel("t (ns)")
axes[2].set_title(f"frame trajectory, max photon number "
                  f"{np.abs(alpha_seq).max() ** 2:.0f}", fontsize=10)

axes[0].set_title("compiled sequence (blue = ECD, orange = rotation)")
fig.tight_layout()
plt.show()

## Simulating the pulses

`simulate_sequence` builds the displaced-frame Hamiltonian of Eq. (S2) as a sum of
Hermitian operators with real, midpoint-sampled coefficients, and propagates it with
`lax.scan`. Midpoint sampling matters here: the Stark-like term $\chi|\alpha|^2 n_t$
reaches tens of rad/us and varies on the pulse timescale.

`truncation_error` is the number to check first. It reports the peak population in the
top few Fock levels *of the frame*; if it is not tiny, raise `n_fock`.

In [ ]:
n_fock_sim = 60
save_every = max(1, seq.n_samples // 120)

t0 = time.time()
res = pulse_simulation.simulate_sequence(
    seq, n_fock=n_fock_sim, n_transmon=2, save_every=save_every, method="eigh"
)
print(f"propagated {seq.n_samples} steps (dim {2 * n_fock_sim}) in {time.time() - t0:.1f} s")
print(f"saved frames        : {res.states.shape[0]}")
print(f"truncation error    : {res.truncation_error():.2e}")
print(f"P(g) at the end     : {res.p_ground()[-1]:.5f}")
print(f"max <n> (lab frame) : {res.photon_number(lab=True).max():.0f}")
print(f"max <n> (in frame)  : {res.photon_number(lab=False).max():.2f}")
print(f"|alpha(T)|          : {abs(res.alpha[-1]):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].plot(res.times * 1e3, res.photon_number(lab=True))
axes[0].set_xlabel("t (ns)")
axes[0].set_ylabel(r"$\langle a^\dagger a \rangle$")
axes[0].set_title("lab-frame photon number")

axes[1].plot(res.times * 1e3, res.p_ground(), label=r"$P(g)$")
axes[1].plot(res.times * 1e3, res.photon_number(lab=False), label=r"$\langle n \rangle$ in frame")
axes[1].set_xlabel("t (ns)")
axes[1].set_title("transmon population and residual occupation")
axes[1].legend()

fig.tight_layout()
plt.show()

### Fidelity

The comparison the experiment makes: project the transmon onto $\lvert g\rangle$ and
compare the cavity state with the target. Three numbers are worth separating.

* the **gate-level** fidelity, the ceiling set by the circuit ansatz at depth $N$;
* the **pulse-level** fidelity, which adds finite pulse durations, Kerr, $\chi'$ and the
  approximate $\theta'$;
* the pulse-level fidelity computed *without* the frame displacement $D(\alpha(T))$,
  which shows how much that one step is worth.

In [ ]:
def pad_to(vec, n):
    out = np.zeros(n, dtype=complex)
    m = min(n, vec.size)
    out[:m] = vec[:m]
    return out / np.linalg.norm(out[:m])


target_sim = pad_to(psi_target, n_fock_sim)

f_pulse = res.fidelity(target_sim, project="ground", apply_frame=True)
f_pulse_noframe = res.fidelity(target_sim, project="ground", apply_frame=False)

print(f"gate-level fidelity            : {result.fidelity:.6f}" if result is not None
      else "gate-level fidelity            : (skipped)")
print(f"pulse-level, with D(alpha(T))  : {f_pulse:.6f}   (infidelity {1 - f_pulse:.2e})")
print(f"pulse-level, without           : {f_pulse_noframe:.6f}")
print(f"cost of ignoring the frame     : {f_pulse - f_pulse_noframe:+.2e}")
print(f"\ntruncation error               : {res.truncation_error():.2e}")

## Wigner movie

Two views of the same trajectory.

**Co-moving**: the Wigner function of the residual state $\lvert\tilde\psi\rangle$ on a
fixed grid. This is where the interesting structure is visible, since the large classical
displacement has been removed.

**Lab frame**: the same arrays, plotted on a grid translated by
$(\sqrt2\,\mathrm{Re}\,\alpha,\ \sqrt2\,\mathrm{Im}\,\alpha)$. This is the physical
picture, in which the state swings out to $|\alpha| \sim 30$ and back.

Mid-sequence the cavity and transmon are deliberately entangled, so `project="traced"`
is the honest choice there; projecting on $\lvert g\rangle$ mid-gate would show an
artificially pure state. The final frame is projected, matching the postselection.

In [ ]:
xvec, yvec, wig, offsets = pulse_simulation.wigner_frames(
    res, x_bound=6.0, y_bound=6.0, grid_points=90, project="traced", frame="lab"
)
print(f"{wig.shape[0]} frames on a {wig.shape[1]}x{wig.shape[2]} grid")

anim = animate_wigner(
    wig, xvec, yvec,
    title="ECD GKP preparation",
    interval=60,
    save_path=f"gkp_ecd_16.mp4",
    dpi=200,
)
HTML(anim.to_jshtml())

In [ ]:
idx = np.linspace(0, wig.shape[0] - 1, 8).astype(int)
fig, axes = plt.subplots(1, 8, figsize=(16, 4))
wmax = np.abs(wig[idx]).max()
for ax, i in zip(axes, idx):
    ax.imshow(
        wig[i],
        origin="lower",
        extent=pulse_simulation.lab_extent(xvec, yvec, offsets[i]),
        cmap="RdBu_r",
        vmin=-wmax,
        vmax=wmax,
        aspect="equal",
    )
    ax.plot([np.sqrt(2) * res.alpha[i].real], [np.sqrt(2) * res.alpha[i].imag], "k+", ms=8)
    ax.set_title(f"t = {res.times[i] * 1e3:.0f} ns\n" r"$|\alpha| = $"
                 f"{abs(res.alpha[i]):.1f}", fontsize=10)
    ax.set_xlabel("q")
axes[0].set_ylabel("p")
fig.suptitle("lab-frame snapshots: the same arrays on translated grids", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
psi_final = res.final_cavity_state(project="ground", apply_frame=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
plot_wigner(wigner=wig[-1], xvec=xvec, yvec=yvec, ax=axes[0],
            title="simulated pulses (transmon traced)")
plot_wigner(state=gkp_0, x_bound=6.0, y_bound=6.0, ax=axes[1],
            title=r"target $|0_L\rangle$")
fig.tight_layout()
plt.show()

print(f"F(cavity | g) = {res.fidelity(target_sim):.6f}")

## The SNAP gate set

A SNAP gate is realized by two number-multiplexed selective $\pi$ pulses whose relative
phase per Fock level sets the imparted phase (Heeres et al., *PRL* **115**, 137002).

The selective pulses must resolve one photon, and one period $2\pi/\chi$ is not enough:
the number-multiplexed components add coherently at the pulse centre, so the drive stops
being a weak selective perturbation and both the $\pi$ rotation and the imparted phases
degrade. `n_selective_periods` defaults to 4, giving $t_\pi \approx 122$ us here and about
$244$ us per SNAP gate (this notebook uses 8, since driving the full truncation
multiplexes more components and so needs longer pulses): nearly *three orders of magnitude* longer than an ECD gate, which
is the entire motivation for ECD control. Errors fall roughly as $1/t_\pi$; the table in
`pulses.snap_waveform` gives measured numbers.

Two further requirements are easy to violate.

*The global sign must be reconciled with the ideal gate.* Two $\pi$ pulses return
$\lvert g\rangle$ to itself with a factor $-e^{i\theta_n}$, while
`gate_optimization._build_snap_sequence` pads unoptimized levels with $\theta_n = 0$, i.e.
with $+1$. Any population above `n_drive` therefore sits $\pi$ out of phase with the
driven block, and with a fifth of the population up there the fidelity collapses to about
$0.34$. `snap_waveform` adds $\pi$ to every second-pulse phase by default
(`match_padded_identity=True`) so that driven levels acquire $+e^{i\theta_n}$ and undriven
ones $+1$: exactly the padded diagonal the optimizer solved for.

*Every occupied Fock level should still be driven*, padding with $\theta_n = 0$. The
correction above removes the $\pi$, but undriven levels are not perfectly untouched: the
off-resonant tails of the multiplexed drive rotate the levels just above `n_drive` by a
few tenths of a radian. Pass `n_drive` equal to the Fock truncation.

*The duration must be an integer multiple of $2\pi/\chi$*, so that the free precession
of level $n$ between the two pulse centres, $\chi n t_\pi$, is a multiple of $2\pi$ and
the two pulses share a phase reference.

*Simulate SNAP in the lab frame.* The displacements here are of order $|\alpha| \sim 2$,
so there is nothing to gain from the displaced frame, and `frame="lab"` avoids the
frame bookkeeping entirely. The waveform is also resampled at 10 ns, since the selective
pulses have bandwidth of order $\chi$ and 1 ns sampling would produce millions of steps.

In [ ]:
from gkp_optimal_control.gate_results import gate_result_path, load_gate_result, save_gate_result
RUN_SNAP_OPTIMIZATION = True

n_fock_snap = 40
n_snap_levels = 8
n_snap_gates = 10

psi_target_snap = pad_to(psi_target, n_fock_snap)
psi_vac_snap = np.zeros(n_fock_snap, dtype=complex)
psi_vac_snap[0] = 1.0

_path = gate_result_path('data/gate_results/ecd_snap_circuit_pulse_compilation', 'snap', n_snap_gates)
if RUN_SNAP_OPTIMIZATION:
    result_snap = optimize_gate_sequence(
        "snap",
        n_gates=n_snap_gates,
        psi_init=psi_vac_snap,
        psi_targ=psi_target_snap,
        n_fock=n_fock_snap,
        n_snap=n_snap_levels,
        loss_type="log_infidelity",
        bounds=GateBounds(max_disp=3.0, n_leak=4),
        optimizer=OptimizerConfig(n_seeds=8, n_adam_iters=1500, seed=1),
        verbose=True,
    )
    save_gate_result(result_snap, _path, initial_state=psi_vac_snap, overwrite=True, source_notebook='ecd_snap_circuit_pulse_compilation')
    print()
    print(result_snap.summary())
    snap_phases = np.asarray(result_snap.params["snap_phases"])
    alphas = np.asarray(result_snap.params["alphas"])
elif _path.exists():
    result_snap = load_gate_result(_path)
    print("loaded cached result from " + str(_path))
else:
    result_snap = None
    rng = np.random.default_rng(0)
    snap_phases = rng.uniform(-np.pi, np.pi, (n_snap_gates, n_snap_levels))
    alphas = rng.normal(size=n_snap_gates + 1) + 1j * rng.normal(size=n_snap_gates + 1)

In [ ]:
# Selectivity degrades as more components are multiplexed, so driving the full
# truncation needs longer pulses than the default 4 periods.
params_snap = params.replace(n_selective_periods=8)

seq_snap = pulses.compile_snap_sequence(
    snap_phases,
    alphas,
    params=params_snap,
    delta=0.0,
    n_drive=n_fock_snap,   # every occupied level, not just the optimized ones
    dt=25e-3,              # 25 ns: the widest carrier is only chi * n_drive
    verbose=True,
)
print()
print(seq_snap.summary())
print(f"\nper-gate SNAP duration: {2 * params_snap.t_snap_selective:.1f} us")
print(f"cavity T1             : {1 / params.kappa:.0f} us")
print(f"mean photons lost over the sequence at <n> = 5: "
      f"{5 * params.kappa * seq_snap.duration:.2f}")
print(f"ECD sequence for comparison: {seq.duration * 1e3:.0f} ns "
      f"({seq_snap.duration / seq.duration:.0f}x shorter)")

In [ ]:
t0 = time.time()
res_snap = pulse_simulation.simulate_sequence(
    seq_snap,
    n_fock=n_fock_snap,
    n_transmon=4,
    save_every=max(1, seq_snap.n_samples // 120),
    frame="lab",
    method="eigh",
)   # tens of thousands of steps: expect a minute or two
print(f"propagated {seq_snap.n_samples} steps in {time.time() - t0:.1f} s")
print(f"truncation error : {res_snap.truncation_error():.2e}")
print(f"P(g) at the end  : {res_snap.p_ground()[-1]:.5f}")
print(f"F(cavity | g)    : {res_snap.fidelity(psi_target_snap, apply_frame=False):.6f}")
if result_snap is not None:
    print(f"gate-level F     : {result_snap.fidelity:.6f}")

In [ ]:
xv_s, yv_s, wig_s, _ = pulse_simulation.wigner_frames(
    res_snap, x_bound=6.0, y_bound=6.0, grid_points=90, project="traced", frame="comoving"
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(res_snap.times, res_snap.p_ground())
axes[0].set_xlabel("t (us)")
axes[0].set_ylabel(r"$P(g)$")
axes[0].set_title("transmon population through the selective pulses")
plot_wigner(wigner=wig_s[-1], xvec=xv_s, yvec=yv_s, ax=axes[1],
            title="SNAP result (lab frame)")
fig.tight_layout()
plt.show()

anim_snap = animate_wigner(wig_s, xv_s, yv_s, title="SNAP preparation", interval=60)
HTML(anim_snap.to_jshtml())

## Notes, conventions and known limits

**Sign of $\beta$.** The compiler realizes $\beta = \alpha_g(T) - \alpha_e(T)$, matching
Eq. (S25), $\mathrm{CD}(\beta) = D(\beta/2)\lvert g\rangle\langle g\rvert +
D(-\beta/2)\lvert e\rangle\langle e\rvert$, and therefore
`gate_optimization.apply_ecd`. Section S4 B of the supplement states the opposite sign
and is inconsistent with both Eq. (S25) and the main text; a single compiled pulse
checked against the ideal gate reaches $|\langle\text{ideal}|\text{sim}\rangle| =
0.999999$ with the convention used here, and essentially zero with the other.

**The frame displacement is not optional.** $|\alpha(T)| \approx 0.02$-$0.03$ looks
negligible but costs several parts in $10^4$ of fidelity, because the residual state is
close to the target and the error adds in amplitude. `final_cavity_state` and `fidelity`
apply $D(\alpha(T))$ by default.

**$\theta'$ is analytic, not measured.** Eq. (S21) is accurate to 10-15% against a full
Hilbert-space simulation. Since $\theta' \approx 0.013\,|\beta|^2$ is at most a few tens
of milliradians, the residual frame error is negligible; if you push to much larger
$|\beta|$ or much deeper circuits, calibrate $\theta'$ against a small simulation instead.

**The propagation is unitary.** `kappa` enters the *compilation* through the semiclassical
trajectories, exactly as in the experiment, but no Lindblad term is propagated.
`include_kappa_force=True` only changes which classical trajectory is factored out. For a
decoherence budget along the lines of Fig. S8, a master-equation path would need adding.

**Rotations are treated as ideal.** The transmon drive is an unselective Gaussian with
bandwidth much larger than $\chi$; the simulation includes its finite duration but the
transmon is truncated to two levels by default. Set `n_transmon=3` or `4` to look for
leakage, which is where the $K \gg \chi$ requirement of the paper bites.

**SNAP is marginal at these parameters, for physical reasons.** With
$\chi/2\pi = 32.8$ kHz the selective pulses run $\sim 122$ us each and a three-gate
sequence takes $\sim 730$ us, against a cavity $T_1$ of $436$ us: photon loss alone
destroys the state, and the unitary simulation here will not show it. Over that duration
the self-Kerr and second-order dispersive shift also accumulate Fock-dependent phases the
ideal gate model omits, of order $0.3$ and $0.45$ rad respectively at $n = 12$, rising to
$0.9$ and $1.3$ rad at $n = 20$. Selectivity gets worse as more levels are multiplexed,
so covering a larger support forces still longer pulses. To exercise the SNAP path
meaningfully, use a device with a larger dispersive shift (Heeres et al. ran at
$\chi/2\pi \approx 2$ MHz, where $t_\pi \sim 3$ us and a sequence fits comfortably inside
$T_1$) rather than the Table S1 numbers, which describe a cavity built for ECD control.

**SNAP carrier sign.** With $H \supset -\chi a^\dagger a\, q^\dagger q$ the
$\lvert g,n\rangle \to \lvert e,n\rangle$ transition sits at detuning $-\chi n$, and a
resonant drive component therefore carries $e^{+i\chi n t}$. The opposite sign leaves only
$n = 0$ resonant: high Fock levels pass through the gate untouched and pick up $+1$
instead of $-e^{i\theta_n}$. The symptom is a per-level phase error that is near zero at
$n = 0$ and grows to $\pi$, with $P(g)$ near unity throughout, since an undriven level and
a level driven by two $\pi$ pulses both end in $\lvert g\rangle$. A per-level SNAP phase cross-check catches it.

**Cross-checks.** An independent NumPy reference implementation cross-checks the same
displaced-frame Hamiltonian, with tests for the displacement and rotation conventions,
one ECD against the ideal gate, the virtual-Z frame rule to machine precision, a full
sequence, and the SNAP phases.
